# Lab 08 — AI_AGG & EXTRACT_ANSWER

**AI_AGG** turns aggregated numbers into narratives. **EXTRACT_ANSWER** finds answers within text passages.

| Function | Returns | Use Case |
|---|---|---|
| `AI_AGG` | Natural-language summary of data | Executive reports, dashboard annotations |
| `EXTRACT_ANSWER` | Answer extracted from source text | Extractive QA over documents |


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_AGG: Numbers to Narratives

In [ ]:
CREATE OR REPLACE VIEW lineitem_sample AS
SELECT
    l_orderkey AS order_key, l_quantity AS quantity,
    l_extendedprice AS extended_price, l_discount AS discount,
    l_returnflag AS return_flag, l_shipdate AS ship_date, l_shipmode AS ship_mode
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM
WHERE l_shipdate BETWEEN '1995-01-01' AND '1995-12-31';

In [ ]:
SELECT SNOWFLAKE.CORTEX.AI_AGG(
    'Summarize the revenue breakdown by shipping mode, highlighting the top performer',
    {
        'ship_mode': ARRAY_AGG(DISTINCT ship_mode),
        'total_revenue': SUM(extended_price),
        'avg_discount': ROUND(AVG(discount), 4),
        'order_count': COUNT(DISTINCT order_key)
    }
) AS summary
FROM lineitem_sample;

In [ ]:
WITH monthly AS (
    SELECT DATE_TRUNC('month', ship_date)::DATE AS month,
        SUM(extended_price) AS revenue, COUNT(*) AS line_count
    FROM lineitem_sample GROUP BY month ORDER BY month
)
SELECT SNOWFLAKE.CORTEX.AI_AGG(
    'Provide a monthly revenue trend analysis for 1995, noting seasonal patterns',
    {'months': ARRAY_AGG(month), 'revenues': ARRAY_AGG(revenue), 'line_counts': ARRAY_AGG(line_count)}
) AS monthly_summary
FROM monthly;

## Step 3 — EXTRACT_ANSWER: Extractive QA

`EXTRACT_ANSWER` pulls the answer to a question directly from source text.
Unlike `AI_EXTRACT` (which extracts named fields), this performs question answering.


In [ ]:
SELECT SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
    'Snowflake was founded in 2012 by Benoit Dageville, Thierry Cruanes, and Marcin Zukowski. The company went public in September 2020 with the largest software IPO in history at the time. Snowflake is headquartered in Bozeman, Montana.',
    'When did Snowflake go public?'
) AS answer;

In [ ]:
CREATE OR REPLACE TABLE qa_docs (doc_id INT, doc_text TEXT);

INSERT INTO qa_docs VALUES
(1, 'Virtual warehouses can be set to auto-suspend after 5 minutes of inactivity. They auto-resume when a query is submitted. Sizes range from X-Small to 6X-Large.'),
(2, 'Snowflake Time Travel allows accessing data as it existed at any point in the last 90 days for Enterprise edition. Standard edition supports up to 1 day.'),
(3, 'Cortex AI functions consume credits based on token usage. COMPLETE uses the most credits, while SENTIMENT and TRANSLATE are more economical.');

SELECT
    doc_id,
    SNOWFLAKE.CORTEX.EXTRACT_ANSWER(doc_text, 'How long is Time Travel retention?') AS answer
FROM qa_docs;

## Key Takeaways

- `AI_AGG` transforms structured aggregates into readable executive summaries.
- Pass a prompt + an OBJECT of aggregate values — ideal for automated reporting.
- `EXTRACT_ANSWER` performs extractive QA — it pulls answers from source text.
- `EXTRACT_ANSWER` ≠ `AI_EXTRACT`: one answers questions, the other extracts named fields.
